# 005_semantic_ledgers — 語意台帳與背景情報表（DDL）

- 用途：方向 C Stage B（規格 §8.3、`_prompt_weekly_extract.md` v1.1 Stage B）的落點。原規劃是 PM_Head 的兩份 md（`semantic_record.md`、`background_intel.md`），Databricks 寫不進 git，改成三張 Delta 表當正本、md 由 `meeting_extracted_sem_stage_b` 產生（D31）。只建結構、不灌資料。
- **正本是 [c04] `table_specs` 與 [c05] `view_specs`**。改欄 = 開新流水號 migration 做 `ALTER TABLE ADD COLUMNS`；View 無狀態，改 [c05] 重跑即可。
- 冪等：表 `CREATE TABLE IF NOT EXISTS`；View `CREATE OR REPLACE VIEW`。不依賴其他表（獨立於 002 / 003）。
- 參數（widgets）：`catalog`、`schema`、`dry_run`（預設 `true`：只驗證 spec 與印出 DDL，不建任何物件）
- 負責人 / 更新日期：（填）/ 2026-10-19

| 物件 | 鍵 | 內容 | 敏感度 |
|---|---|---|---|
| `g_semantic_record` | `sem_id`（SEM-001…） | 定案的業務語意：主題 / 定義 / 投影代號 / app / 第一筆證據；`status` 與 `synced_to` 由 PM 維護 | S0–S1 |
| `g_semantic_record_evidence` | `(sem_id, item_key)` | 每條定案語意的證據（quote ＋ date ＋ source），dup_of 補證據就是加一列 | S1–S2 |
| `g_background_intel` | `intel_id`（INT-001…） | 背景情報，一列一主題（app ＋ term），同主題再出現就更新並刷新 `as_of`；`status` 由 PM 維護 | S1–S2 |
| `v_semantic_record_q` | — | 定案區（status ≠ retired） | S0–S1 |
| `v_background_intel_q` | — | 情報 ＋ `quarters_old` / `retire_candidate`（as_of 距今超過兩季） | S1–S2 |
| `b_meeting_migration_log` | `(slug, run_at)` | 執行紀錄（IF NOT EXISTS） | – |

**不在本 migration**：Stage B 的彙整邏輯（另一支 notebook）；Sales Brief 鏡頭（004）。


In [ ]:
# [c01] params
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
dry_run = dbutils.widgets.get("dry_run") == "true"
assert catalog and schema, "catalog / schema 不可為空"
print({"catalog": catalog, "schema": schema, "dry_run": dry_run})

In [ ]:
# [c02] imports
# 本 cell 與 [c03]～[c05] 不碰 spark / dbutils，可被 tests/ 載入。
import json
import re

SLUG = "005_semantic_ledgers"
COL_RE = r"^[a-z][a-z0-9_]*$"
TYPE_RE = r"^(STRING|BIGINT|INT|DOUBLE|BOOLEAN|DATE|TIMESTAMP|ARRAY<STRING>)$"

# Stage B 依賴的欄一定要有
REQUIRED_COLS = {
    "g_semantic_record": ["sem_id", "term", "summary", "route_hint", "app", "status", "n_evidence"],
    "g_semantic_record_evidence": ["sem_id", "item_key", "quote", "date", "source"],
    "g_background_intel": ["intel_id", "app", "term", "term_norm", "summary", "as_of", "status", "n_updates"],
}
# S3 欄位：任何 View 不得暴露（與 002 / 003 同一份）
S3_COLS = {
    "display_name_raw", "name_zh", "name_en", "email", "emp_id", "account",
    "forwarder_name_zh", "forwarder_emp_id", "forwarder_account",
    "mail_to_raw", "mail_cc_raw", "mail_bcc_raw", "body_display",
}
RETIRE_QUARTERS = 2  # 規格：as_of 距今超過兩季 → 退場候選（只標示，PM 決定）

In [ ]:
# [c03] pure_ddl
# spec → SQL 字串的純函式；不碰 spark。測試在本機用同一份 spec 建空表跑 View 的 SELECT。
def q(s: str | None) -> str:
    """SQL 單引號字串常值；None → ''。"""
    return "'" + (s or "").replace("\\", "\\\\").replace("'", "\\'") + "'"


def build_create_table(fq: str, spec: dict) -> str:
    """CREATE TABLE IF NOT EXISTS；鍵欄 NOT NULL；有 cluster_by 才加 CLUSTER BY（liquid clustering）。"""
    keys = set(spec["key"])
    cols = ",\n".join(
        f"    {name} {typ}{' NOT NULL' if name in keys else ''} COMMENT {q(comment)}"
        for name, typ, comment in spec["columns"]
    )
    cluster = f"\nCLUSTER BY ({', '.join(spec['cluster_by'])})" if spec.get("cluster_by") else ""
    return f"CREATE TABLE IF NOT EXISTS {fq} (\n{cols}\n) USING DELTA{cluster}\nCOMMENT {q(spec['comment'])}"


def build_select(spec: dict, fq) -> str:
    """View 的查詢本體：spec['select'] = [(expr, alias, comment)]，spec['body'](fq) 回傳 FROM 起的 SQL。
    fq(table_name) -> 可查詢的名稱；正式是三段式，測試用 temp view 名。"""
    select = ",\n".join(f"    {expr} AS {alias}" for expr, alias, _ in spec["select"])
    return f"SELECT\n{select}\n{spec['body'](fq)}"


def build_create_view(fq_view: str, spec: dict, fq) -> str:
    """CREATE OR REPLACE VIEW，欄位清單帶 COMMENT（Genie / 人看的語意）。"""
    col_list = ",\n".join(f"    {alias} COMMENT {q(comment)}" for _, alias, comment in spec["select"])
    return (
        f"CREATE OR REPLACE VIEW {fq_view} (\n{col_list}\n)\nCOMMENT {q(spec['comment'])}\nAS\n"
        + build_select(spec, fq)
    )


def schema_ddl(spec: dict) -> str:
    """欄位 → 'a BIGINT, b STRING'（測試建空表、[c30] 比對型別用）。"""
    return ", ".join(f"{n} {t}" for n, t, _ in spec["columns"])


def add_columns_sql(fq: str, spec: dict, missing: list[str]) -> str:
    """[c30] 發現缺欄時印出的補欄語句（不自動執行；要補就開新流水號 migration）。"""
    by_name = {n: (t, c) for n, t, c in spec["columns"]}
    parts = ", ".join(f"{n} {by_name[n][0]} COMMENT {q(by_name[n][1])}" for n in missing)
    return f"ALTER TABLE {fq} ADD COLUMNS ({parts})"


def validate_specs(tables: dict, views: dict) -> list[str]:
    """回傳錯誤訊息；空 list = 通過。"""
    errs = []
    for t, spec in tables.items():
        names = [c[0] for c in spec["columns"]]
        dup = sorted({n for n in names if names.count(n) > 1})
        if dup:
            errs.append(f"{t} 欄位重複：{dup}")
        for n, typ, comment in spec["columns"]:
            if not re.match(COL_RE, n):
                errs.append(f"{t}.{n} 欄名格式不符（小寫英數底線）")
            if not re.match(TYPE_RE, typ):
                errs.append(f"{t}.{n} 型別不在允許清單：{typ}")
            if not comment:
                errs.append(f"{t}.{n} 沒有 COMMENT")
        for k in list(spec["key"]) + list(spec.get("cluster_by", [])):
            if k not in names:
                errs.append(f"{t} key / cluster_by 欄 {k} 不存在")
        for r in REQUIRED_COLS.get(t, []):
            if r not in names:
                errs.append(f"{t} 缺必要欄 {r}")
        if not spec.get("comment"):
            errs.append(f"{t} 沒有表 COMMENT")
    for v, spec in views.items():
        aliases = [a for _, a, _ in spec["select"]]
        dup = sorted({a for a in aliases if aliases.count(a) > 1})
        if dup:
            errs.append(f"{v} 輸出欄重複：{dup}")
        leak = sorted(S3_COLS & set(aliases))
        if leak:
            errs.append(f"{v} 暴露 S3 欄：{leak}")
        for _, a, c in spec["select"]:
            if not re.match(COL_RE, a):
                errs.append(f"{v}.{a} 欄名格式不符")
            if not c:
                errs.append(f"{v}.{a} 沒有 COMMENT")
        if not spec.get("comment"):
            errs.append(f"{v} 沒有 View COMMENT")
    return errs

In [ ]:
# [c04] table_specs
# 正本。欄位 = (名稱, 型別, COMMENT)。鍵欄自動 NOT NULL。
# 三張表 = _prompt_weekly_extract.md v1.1 Stage B 兩份 md 的表格版（D31）：
#   g_semantic_record          = semantic_record.md 定案區（一列一個 SEM id）
#   g_semantic_record_evidence = 每個 SEM 的證據列（dup_of 補證據 = 加一列）
#   g_background_intel         = background_intel.md 各 app 節（一列一主題，同主題更新不新增）
TABLES = {
    "g_semantic_record": {
        "comment": "業務語意台帳定案區（方向 C Stage B 產出）：一列一個 SEM id；PM 審過（verified = y）的 semantic 條目在此定案。status / synced_to / pm_note 由 PM 維護，Stage B 不覆蓋。不含姓名 email。正本：005_semantic_ledgers [c04]。",
        "key": ["sem_id"],
        "cluster_by": [],
        "columns": [
            ("sem_id", "STRING", "SEM-001 起流水號（Stage B 發）"),
            ("term", "STRING", "主題短語（取自定案條目的 term）"),
            ("summary", "STRING", "定義 / 規則（取自定案條目的 summary）"),
            ("route_hint", "STRING", "投影代號（SemanticLayerPlan §1.3）：pa:semantics / skill:multi_genie / de:主控 / genie:<space> / brief:過濾卡 / mv:<view>；PM 在 review_note 寫了代號就以 review_note 為準"),
            ("app", "STRING", "TV / NB / MNT / ALL"),
            ("quote", "STRING", "第一筆證據的逐字原文（已 token 化）"),
            ("date", "DATE", "第一筆證據的日期"),
            ("source", "STRING", "第一筆證據的場合"),
            ("first_item_key", "STRING", "定案來源條目的 item_key（回溯鏡頭表）"),
            ("n_evidence", "INT", "證據列數（g_semantic_record_evidence）"),
            ("last_evidence_date", "DATE", "最新證據日期"),
            ("status", "STRING", "confirmed / retired；PM 維護，Stage B 不覆蓋"),
            ("synced_to", "STRING", "已編譯進哪些 prompt（SemanticLayerPlan §1.2 第 4 步，人填）；NULL = 尚未投影"),
            ("pm_note", "STRING", "定案時鏡頭表的 review_note 複製；之後 PM 可改"),
            ("stage_b_version", "STRING", "lineage"),
            ("created_at", "TIMESTAMP", "定案時間"),
            ("updated_at", "TIMESTAMP", "最近一次 Stage B 更新（補證據）"),
        ],
    },
    "g_semantic_record_evidence": {
        "comment": "語意台帳的證據列：每個 SEM id 的每筆佐證（quote ＋ date ＋ source）。定案那條是第一列，之後 dup_of 命中的條目各加一列。不含姓名 email。正本：005 [c04]。",
        "key": ["sem_id", "item_key"],
        "cluster_by": ["sem_id"],
        "columns": [
            ("sem_id", "STRING", "對應 g_semantic_record"),
            ("item_key", "STRING", "來源條目（鏡頭表 / 中立層的 item_key）"),
            ("item_id", "STRING", "加入時的中立條目 id"),
            ("quote", "STRING", "逐字原文（已 token 化）"),
            ("date", "DATE", "原始訊號日期"),
            ("source", "STRING", "場合＋段日期＋發言方"),
            ("app", "STRING", "條目的 app"),
            ("customer", "STRING", "條目的客戶"),
            ("week", "STRING", "條目的 ISO 週"),
            ("added_at", "TIMESTAMP", "加入時間"),
        ],
    },
    "g_background_intel": {
        "comment": "背景情報台帳（方向 C Stage B 產出）：一列一主題（app ＋ term），PM 審過的 intel 條目併入；同主題再出現 → 更新 summary / as_of / source、n_updates + 1，不新增。as_of 距今超過兩季為退場候選（v_background_intel_q.retire_candidate），status 由 PM 改。不含姓名 email。正本：005 [c04]。",
        "key": ["intel_id"],
        "cluster_by": ["app"],
        "columns": [
            ("intel_id", "STRING", "INT-001 起流水號（Stage B 發）"),
            ("app", "STRING", "TV / NB / MNT / ALL（background_intel.md 的節）"),
            ("term", "STRING", "主題（background_intel.md 的「主題」欄）"),
            ("term_norm", "STRING", "term 正規化（NFKC、去標點空白、小寫）；同 app 同 term_norm = 同主題"),
            ("summary", "STRING", "判讀框架（含 so-what）；最新一次的"),
            ("as_of", "STRING", "有效基準 yyyy-MM 或 yyyy-Qn；更新時取較新的"),
            ("date", "DATE", "最新證據日期"),
            ("source", "STRING", "最新證據場合"),
            ("item_key", "STRING", "最新證據條目的 item_key"),
            ("first_item_key", "STRING", "第一次建立的條目 item_key"),
            ("n_updates", "INT", "被更新次數（含建立 = 1）"),
            ("status", "STRING", "active / unreviewed / retired：unreviewed = intel 免逐條審時由 Stage B 自動併入（D32），PM 在鏡頭表標 y → active、全部標 n → retired（Stage B 同步）；active / retired 由 PM 維護，Stage B 不覆蓋"),
            ("pm_note", "STRING", "鏡頭表 review_note 複製（最新一次）；之後 PM 可改"),
            ("stage_b_version", "STRING", "lineage"),
            ("created_at", "TIMESTAMP", "建立時間"),
            ("updated_at", "TIMESTAMP", "最近更新"),
        ],
    },
    "b_meeting_migration_log": {
        "comment": "會議萃取 migration 執行紀錄（只增不改）",
        "key": ["slug", "run_at"],
        "columns": [
            ("slug", "STRING", "migration 流水號＋名稱（例 002_meeting_neutral_tables）"),
            ("run_at", "TIMESTAMP", "執行時間"),
            ("run_by", "STRING", "current_user()"),
            ("result", "STRING", "JSON：各物件 created / exists / replaced"),
            ("note", "STRING", "備註"),
        ],
    },
}

In [ ]:
# [c05] view_specs
# 正本。select = [(expr, 輸出欄名, COMMENT)]；body(fq) 回傳 FROM 起的 SQL，fq(表名) 給完整名稱。
#   v_semantic_record_q  = 定案區（status ≠ retired）
#   v_background_intel_q = 情報 ＋ quarters_old / retire_candidate（as_of 距今超過 RETIRE_QUARTERS 季；status 有 active / unreviewed / retired，D32）
# as_of → 季索引：yyyy-Qn → 年×4 + n − 1；yyyy-MM → 年×4 + (月−1) DIV 3；格式不符 → NULL（不算退場候選）。
_AS_OF_Q = r"""CASE
        WHEN s.as_of RLIKE '^[0-9]{4}-Q[1-4]$' THEN cast(substr(s.as_of, 1, 4) AS INT) * 4 + cast(substr(s.as_of, 7, 1) AS INT) - 1
        WHEN s.as_of RLIKE '^[0-9]{4}-(0[1-9]|1[0-2])$' THEN cast(substr(s.as_of, 1, 4) AS INT) * 4 + (cast(substr(s.as_of, 6, 2) AS INT) - 1) DIV 3
        END"""
_NOW_Q = "year(current_date()) * 4 + quarter(current_date()) - 1"
_QUARTERS_OLD = f"({_NOW_Q}) - ({_AS_OF_Q})"


def _record_body(fq):
    return f"""FROM {fq('g_semantic_record')} s
WHERE coalesce(s.status, 'confirmed') <> 'retired'"""


def _intel_body(fq):
    return f"""FROM {fq('g_background_intel')} s"""


VIEWS = {
    "v_semantic_record_q": {
        "comment": "業務語意台帳定案區（status ≠ retired）：Stage B 產 semantic_record.md、prompt 編譯、Genie 查口徑用。不含姓名 email。正本：005 [c05]。",
        "body": _record_body,
        "select": [
            ("s.sem_id", "sem_id", "SEM id"),
            ("s.app", "app", "TV / NB / MNT / ALL"),
            ("s.term", "term", "主題"),
            ("s.summary", "summary", "定義 / 規則"),
            ("s.route_hint", "route_hint", "投影代號"),
            ("s.quote", "quote", "第一筆證據原文"),
            ("s.date", "date", "第一筆證據日期"),
            ("s.source", "source", "第一筆證據場合"),
            ("s.n_evidence", "n_evidence", "證據數"),
            ("s.last_evidence_date", "last_evidence_date", "最新證據日期"),
            ("s.status", "status", "confirmed / retired"),
            ("s.synced_to", "synced_to", "已投影到哪些 prompt（人填）"),
            ("s.pm_note", "pm_note", "PM 備註"),
            ("s.first_item_key", "first_item_key", "回溯鏡頭表"),
            ("s.created_at", "created_at", "定案時間"),
            ("s.updated_at", "updated_at", "最近更新"),
        ],
    },
    "v_background_intel_q": {
        "comment": "背景情報台帳 ＋ 退場判斷：quarters_old = as_of 距今幾季，retire_candidate = active 且超過兩季（只標示，PM 決定）。Stage B 產 background_intel.md 用。不含姓名 email。正本：005 [c05]。",
        "body": _intel_body,
        "select": [
            ("s.intel_id", "intel_id", "INT id"),
            ("s.app", "app", "TV / NB / MNT / ALL"),
            ("s.term", "term", "主題"),
            ("s.summary", "summary", "判讀框架"),
            ("s.as_of", "as_of", "有效基準"),
            (_QUARTERS_OLD, "quarters_old", "as_of 距今幾季（格式不符 NULL）"),
            (f"coalesce(s.status, 'active') <> 'retired' AND ({_QUARTERS_OLD}) > {RETIRE_QUARTERS}", "retire_candidate",
             "退場候選：未 retired（active / unreviewed）且 as_of 超過兩季"),
            ("s.date", "date", "最新證據日期"),
            ("s.source", "source", "最新證據場合"),
            ("s.item_key", "item_key", "最新證據條目"),
            ("s.n_updates", "n_updates", "更新次數"),
            ("s.status", "status", "active / retired"),
            ("s.pm_note", "pm_note", "PM 備註"),
            ("s.created_at", "created_at", "建立時間"),
            ("s.updated_at", "updated_at", "最近更新"),
        ],
    },
}

In [ ]:
# [c10] build_ddl
errors = validate_specs(TABLES, VIEWS)
for e in errors:
    print("ERROR", e)
assert not errors, f"spec 驗證失敗 {len(errors)} 筆，修 [c04] / [c05] 後重跑"


def fq(name: str) -> str:
    return f"{catalog}.{schema}.{name}"


table_ddl = {t: build_create_table(fq(t), s) for t, s in TABLES.items()}
view_ddl = {v: build_create_view(fq(v), s, fq) for v, s in VIEWS.items()}
existing = {t: spark.catalog.tableExists(fq(t)) for t in list(TABLES) + list(VIEWS)}
print("既有物件：", {k: v for k, v in existing.items() if v} or "無")
print("將建立：", [k for k, v in existing.items() if not v])
if dry_run:
    print("dry_run = true：只印 DDL，不建任何物件。確認後把 dry_run 改 false 重跑。\n")
    for ddl in list(table_ddl.values()) + list(view_ddl.values()):
        print(ddl, end=";\n\n")

In [ ]:
# [c20] create_tables
# 一次性 DDL；IF NOT EXISTS 讓重跑無害。既有表的欄位差異由 [c30] 報告，不在這裡 ALTER。
if not dry_run:
    for t, ddl in table_ddl.items():
        spark.sql(ddl)
        print(("exists " if existing[t] else "created") + " " + fq(t))

In [ ]:
# [c21] create_views
# View 無狀態：CREATE OR REPLACE，改 [c05] 後重跑即生效。必須在 [c20] 之後（View 引用四張表）。
if not dry_run:
    for v, ddl in view_ddl.items():
        spark.sql(ddl)
        print(("replaced" if existing[v] else "created ") + " " + fq(v))

In [ ]:
# [c22] write_log
# 每次正式執行（dry_run = false）寫一列 b_meeting_migration_log；只 INSERT，不改舊列。
if not dry_run:
    result = {
        "tables": {t: ("exists" if existing[t] else "created") for t in TABLES},
        "views": {v: ("replaced" if existing[v] else "created") for v in VIEWS},
    }
    spark.sql(
        f"INSERT INTO {fq('b_meeting_migration_log')} "
        "SELECT :slug, current_timestamp(), current_user(), :result, :note",
        args={"slug": SLUG, "result": json.dumps(result, ensure_ascii=False), "note": None},
    )
    print("logged", result)

In [ ]:
# [c30] check_target
# 比對實際表欄位 vs spec：缺欄（表早已存在、spec 後來加欄）→ 印補欄語句並 fail；型別不同 → 警告。
# View：能查、輸出欄 = spec、不含 S3 欄。
if not dry_run:
    problems = []
    for t, spec in TABLES.items():
        actual = {f.name: f.dataType.simpleString() for f in spark.table(fq(t)).schema.fields}
        missing = [n for n, _, _ in spec["columns"] if n not in actual]
        if missing:
            problems.append(f"{t} 缺欄 {missing}。補欄（開新流水號執行）：\n  {add_columns_sql(fq(t), spec, missing)}")
        for n, typ, _ in spec["columns"]:
            if n in actual and actual[n] != typ.lower():
                print(f"WARN {t}.{n} 型別 spec={typ.lower()} actual={actual[n]}")
        extra = [n for n in actual if n not in {c[0] for c in spec["columns"]}]
        if extra:
            print(f"INFO {t} 有 spec 以外的欄（不處理）：{extra}")
    for v, spec in VIEWS.items():
        cols = spark.table(fq(v)).columns
        want = [a for _, a, _ in spec["select"]]
        if cols != want:
            problems.append(f"{v} 輸出欄與 spec 不同：{cols}")
        leak = sorted(S3_COLS & set(cols))
        if leak:
            problems.append(f"{v} 暴露 S3 欄：{leak}")
        spark.table(fq(v)).limit(1).collect()  # 能執行（表空也要能跑）
    for p in problems:
        print("PROBLEM", p)
    assert not problems, f"{len(problems)} 個問題"
    print("tables:", {t: len(spark.table(fq(t)).columns) for t in TABLES})
    print("views :", {v: len(spark.table(fq(v)).columns) for v in VIEWS})